In [ ]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Soma total de vendas por dia , ordens com status de 'Delivered' e estate = NY           #              #    Origem camada silver: customer                                                         #
#                          orders                                                           # 
#                          product                                                          #
#    Destino camada gold: sales_ny                                                          #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela silver                              #
# 2. Criar uma tabela gold com os dados da tabela silver                                    #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [ ]:
import os
import sys

# Procura automaticamente em qual pasta acima está a 'funcoes_diversas'
caminho = os.getcwd()
while caminho != os.path.dirname(caminho):
    if os.path.exists(os.path.join(caminho, "funcoes_diversas")):
        if caminho not in sys.path:
            sys.path.insert(0, caminho)
        break
    caminho = os.path.dirname(caminho)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria

In [ ]:
mapa_criado = gerar_tabela_temporaria(["orders"],"silver")

In [ ]:
df_sales_ny_gold = spark.sql("""
    select ORD.shipped_date,
           round(sum(ORD.total_sale),2) as total_sale        
    from tmp_silver_orders as ORD
    where 1 = 1
      and ORD.state = 'NY'
      and ORD.status_descricao = 'Delivered'
      group by ORD.shipped_date     
""")

In [ ]:
gold_path = buscar_caminho("gold")
# transformando consulta em uma tabela Silver 
df_sales_ny_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{gold_path}/sales_ny")